# 05 — Model Interpretation

Feature importance analysis for the best-performing model.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import yaml
from pathlib import Path

cfg = yaml.safe_load(open('../configs/config.yaml'))
model = joblib.load('../models/best_model.joblib')
with open('../models/model_metadata.json') as f:
    meta = json.load(f)

feature_cols = meta['feature_cols']
model_type = meta['model_type']
print(f"Model: {model_type}")
print(f"Features: {len(feature_cols)}")


## Built-in Feature Importance

In [ ]:
from src.evaluation.plots import plot_feature_importance
FIG_INTERP = Path('../reports/figures/interpretation')
FIG_INTERP.mkdir(parents=True, exist_ok=True)

final_step = model.named_steps.get('model', None)

if hasattr(final_step, 'feature_importances_'):
    importances = final_step.feature_importances_
    plot_feature_importance(feature_cols, importances, model_type, FIG_INTERP)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    idx = np.argsort(importances)[::-1][:20]
    ax.barh([feature_cols[i] for i in idx][::-1], importances[idx][::-1], color='#42A5F5')
    ax.set_title(f'Feature Importance - {model_type}')
    ax.set_xlabel('Importance Score')
    plt.tight_layout(); plt.show()
    
    print("Top 10 most important features:")
    for i in idx[:10]:
        print(f"  {feature_cols[i]:<30} {importances[i]:.6f}")
elif hasattr(final_step, 'coef_'):
    coefs = np.abs(final_step.coef_)
    print("Linear Regression |coefficient| magnitudes (standardised):")
    idx = np.argsort(coefs)[::-1][:10]
    for i in idx:
        print(f"  {feature_cols[i]:<30} |coef|={coefs[i]:.4f}")
else:
    print(f"Model {model_type} does not have standard feature_importances_.")


## Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

df_proc = pd.read_csv('../data/processed/hourly_processed.csv', index_col=0, parse_dates=True)
df_proc.index = pd.to_datetime(df_proc.index, utc=True)

from src.features.build_features import build_features, get_feature_cols
from src.models.train import chronological_split

feat = build_features(df_proc, lag_hours=meta['lag_hours'],
                      rolling_windows=meta['rolling_windows'], horizon=meta['horizon_hours'])
_, test = chronological_split(feat, cfg['model']['train_fraction'])
X_test = test[feature_cols].dropna()
y_test = test.loc[X_test.index, 'target']

print(f"Running permutation importance on {len(X_test)} test samples...")
r = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_imp = r.importances_mean

idx_perm = np.argsort(perm_imp)[::-1][:15]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([feature_cols[i] for i in idx_perm][::-1], perm_imp[idx_perm][::-1], color='#66BB6A')
ax.set_title(f'Permutation Importance - {model_type} (test set)')
ax.set_xlabel('Mean decrease in MAE')
plt.tight_layout(); plt.show()

print("Top 10 features by permutation importance:")
for i in idx_perm[:10]:
    print(f"  {feature_cols[i]:<30} {perm_imp[i]:.6f} +/- {r.importances_std[i]:.6f}")


## Interpretation Notes

- **Feature importance ≠ causation**: High importance means the model relies on the feature
  heavily for predictions, not that the feature causes PM2.5 to change.
- **Correlated features**: When features are correlated (e.g., pm25_lag0 and pm25_lag1h),
  importance can be distributed arbitrarily between them. Remove one and the other gains importance.
- **Permutation importance** is more reliable than built-in importance as it measures
  impact on actual test predictions, not training split quality.
- **Lag features dominate**: As expected in time-series forecasting, recent PM2.5 values
  (pm25_lag0, pm25_lag1h, pm25_roll24h_mean) tend to be the most predictive features
  for 24h-ahead forecasting.
